In [5]:
from architector import convert_io_molecule,view_structures
from architector.io_align_mol import reorder_align_rmsd
from architector.io_calc import CalcExecutor
from architector.io_molecule import convert_io_molecule
import architector.io_ptable as io_ptable

from ase.constraints import Hookean,ExternalForce
import numpy as np
import copy
from mace.calculators import mace_mp
from ase import build


def check_bonds(mol,
                bonds_breaking,
                bonds_forming,
                breaking_cutoff,
                forming_cutoff): # Convergence check function
    dists = mol.ase_atoms.get_all_distances()
    anums = mol.ase_atoms.get_atomic_numbers()
    goods = []
    for inds in bonds_breaking:
        cutoff_dist = (io_ptable.rcov1[anums[inds[0]]] + io_ptable.rcov1[anums[inds[1]]])*breaking_cutoff
        actual_dist = dists[inds[0]][inds[1]]
        if actual_dist > cutoff_dist:
            goods.append(True)
        else:
            goods.append(False)
    for inds in bonds_forming:
        cutoff_dist = (io_ptable.rcov1[anums[inds[0]]] + io_ptable.rcov1[anums[inds[1]]])*forming_cutoff
        actual_dist = dists[inds[0]][inds[1]]
        if actual_dist < cutoff_dist:
            goods.append(True)
        else:
            goods.append(False)
    return np.all(goods)


In [3]:
mol_init = convert_io_molecule('r3.xyz')
mol_final = convert_io_molecule('p3.xyz')
view_structures([mol_init,mol_final])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
mol_init.detect_charge_spin()
print(mol_init.uhf,mol_init.charge)
mol_final.detect_charge_spin()
print(mol_final.uhf,mol_final.charge)

0 0
0 0


In [5]:
mol1_relaxed = CalcExecutor(mol_init, relax=False).mol
view_structures(mol1_relaxed)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
mol2_relaxed = CalcExecutor(mol_final, relax=True).mol
view_structures(mol2_relaxed)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
mol_final_ats = reorder_align_rmsd(mol_init.ase_atoms, mol_final.ase_atoms)
mol_final = convert_io_molecule(mol_final_ats)
view_structures([mol_init, mol_final], labelinds=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
breaking_cutoff=1.5 # When a bond is breaking, what the distance should be
forming_cutoff=1.2 # When a bond is forming, what the distance should be (Angstroms)
start_force_constant=0.05 # eV/angstrom
force_increment=0.05 # How fast to ramp up the force constant
structure_match_fconst=0.001 # Add springs to force the inial geometry towards the final geometry?
method='GFN2-xTB' # XTB
max_steps=4 # Steps/opimization iteration
fmax_opt=0.1 # Cutoff for the maximum force.
initial = mol_init # Initial configuration
final = mol_final # Final Configuration.
mol1 = convert_io_molecule(initial)
mol2 = convert_io_molecule(final)
mol1.create_mol_graph()
mol2.create_mol_graph()
# Find the formed bonds
bonds_forming = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == 1)) if x[0] < x[1]]
# Find the broken bonds
bonds_breaking = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == -1)) if x[0] < x[1]]
fconst = start_force_constant
save_trajectory = []
opt_mol = copy.deepcopy(mol1)
keep_going = True # Exit flag for loop
while keep_going:
    print('Running Fconst = {}'.format(fconst))
    opt_mol.ase_atoms.set_constraint()
    constraints = []
    for inds in bonds_forming: # Add hookean constants
        constraint = ExternalForce(inds[0], inds[1], -fconst)
        constraints.append(constraint)
    for inds in bonds_breaking:
        constraint = ExternalForce(inds[0], inds[1], fconst)
        constraints.append(constraint)
    for ind in range(mol1.graph.shape[0]): # Add distance setting 
        constraint = Hookean(ind, mol2.ase_atoms.positions[ind],
                                k=structure_match_fconst,
                                rt=0.1)
        constraints.append(constraint)
    opt_mol.ase_atoms.set_constraint(constraints)
    tmpopt = CalcExecutor(opt_mol,
                            method=method,
                            relax=True,
                            fmax=fmax_opt,
                            maxsteps=max_steps,
                            save_trajectories=True,
                            use_constraints=True)
    tmpopt.mol.ase_atoms.calc = None
    save_trajectory += tmpopt.trajectory
    # save_trajectory.append(copy.deepcopy(tmpopt.mol))
    opt_mol = tmpopt.mol
    good = check_bonds(opt_mol, bonds_breaking, bonds_forming,
                        breaking_cutoff, forming_cutoff)
    if good:
        keep_going = False
    else:
        fconst += force_increment

Running Fconst = 0.05
Running Fconst = 0.1
Running Fconst = 0.15000000000000002
Running Fconst = 0.2


In [9]:
view_structures(save_trajectory,trajectory=True,interval=400,w=600,h=600)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
from ase.io import read # Read in the initial and final molecules.
# mols = read('geodesic_paths/R19-1_path.xyz',index=':')
mols = read('geodesic_paths/R30_path.xyz',index=':')
mol_init = convert_io_molecule(mols[0])
mol_final = convert_io_molecule(mols[-1])
view_structures([mol_init,mol_final])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
mol_init.detect_charge_spin()
print(mol_init.uhf,mol_init.charge)
mol_final.detect_charge_spin()
print(mol_final.uhf,mol_final.charge)
# Get the charge/spin parameters

0 0
0 -2


In [13]:
class AFIRPushConstraint():

    def __init__(self, a1, a2, f_ext, max_dist = None):
        self.indices = [a1, a2]
        self.external_force = f_ext
        self.max_dist = max_dist

    def get_removed_dof(self, atoms):
        return 0

    def adjust_positions(self, atoms, new):
        pass

    def adjust_forces(self, atoms, forces):
        dist = np.subtract.reduce(atoms.positions[self.indices])
        if self.max_dist is not None and np.linalg.norm(dist) < self.max_dist:
            force = self.external_force * dist / np.linalg.norm(dist)
            forces[self.indices] += (force, -force)

    def adjust_potential_energy(self, atoms):
        dist = np.subtract.reduce(atoms.positions[self.indices])
        if self.max_dist is not None and np.linalg.norm(dist) < self.max_dist:
            return -np.linalg.norm(dist) * self.external_force
        else:
            return 0

In [14]:
breaking_cutoff=1.5 # When a bond is breaking, what the distance should be
forming_cutoff=1.2 # When a bond is forming, what the distance should be (Angstroms)
start_force_constant=0.1 # eV/angstrom
force_increment=0.2 # How fast to ramp up the force constant
structure_match_fconst=0.000 # Add springs to force the inial geometry towards the final geometry?
# method='GFN1-xTB' # XTB
# method='GFN-FF'
calc = mace_mp(model="medium", dispersion=True, default_dtype="float64")
method='custom'
max_steps=50 # Steps/opimization iteration
fmax_opt=0.15 # Cutoff for the maximum force.
initial = mol_init # Initial configuration
final = mol_final # Final Configuration.
mol1 = convert_io_molecule(initial)
mol2 = convert_io_molecule(final)
mol1.create_mol_graph()
mol2.create_mol_graph()
# Find the formed bonds
bonds_forming = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == 1)) if x[0] < x[1]]
# Find the broken bonds
bonds_breaking = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == -1)) if x[0] < x[1]]
print(bonds_breaking)
print(bonds_forming)
fconst = start_force_constant
save_trajectory = []
opt_mol = copy.deepcopy(mol1)
keep_going = True # Exit flag for loop
nstep = 0 
while keep_going:
    print('Running Fconst = {}'.format(fconst))
    opt_mol.ase_atoms.set_constraint()
    nstep += 1
    constraints = []
    for inds in bonds_forming: # Add hookean constants
        constraint = ExternalForce(inds[0], inds[1], -fconst)
        constraints.append(constraint)
    for inds in bonds_breaking:
        constraint = AFIRPushConstraint(inds[0], inds[1], fconst, 5.0)
        constraints.append(constraint)
    #for ind in range(mol1.graph.shape[0]): # Add distance setting 
    #    constraint = Hookean(ind, mol2.ase_atoms.positions[ind],
    #                            k=structure_match_fconst,
    #                            rt=0.1)
    #    constraints.append(constraint)
    opt_mol.ase_atoms.set_constraint(constraints)
    tmpopt = CalcExecutor(opt_mol,
                            method=method,
                            relax=True,
                            fmax=fmax_opt,
                            maxsteps=max_steps,
                            save_trajectories=True,
                            calculator=calc,
                            use_constraints=True)
    tmpopt.mol.ase_atoms.calc = None
    save_trajectory.extend(copy.deepcopy(tmpopt.trajectory))
    opt_mol = tmpopt.mol
    good = check_bonds(opt_mol, bonds_breaking, bonds_forming,
                        breaking_cutoff, forming_cutoff)
    if good:
        keep_going = False
    else:
        fconst += force_increment

Using local medium Materials Project MACE model for MACECalculator /Users/mgt16/software/mace/mace/calculators/foundations_models/2023-12-03-mace-mp.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
Using TorchDFTD3Calculator for D3 dispersion corrections (see https://github.com/pfnet-research/torch-dftd)
[(29, 42)]
[(8, 9)]
Running Fconst = 0.1
Running Fconst = 0.30000000000000004
Running Fconst = 0.5
Running Fconst = 0.7
Running Fconst = 0.8999999999999999


In [15]:
view_structures(save_trajectory,trajectory=True,interval=200,w=500,h=500)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
view_structures(save_trajectory,trajectory=True,interval=200,w=500,h=500)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [29]:
len(save_trajectory)

188

In [1]:

atoms = build.molecule('H2O')
calc = mace_mp(model="medium", dispersion=True, default_dtype="float64")
atoms.calc = calc
print(atoms.get_potential_energy())

Using local medium Materials Project MACE model for MACECalculator /Users/mgt16/software/mace/mace/calculators/foundations_models/2023-12-03-mace-mp.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/Users/mgt16/mambaforge/lib/python3.10/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/Users/mgt16/mambaforge/lib/python3.10/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/Users/mgt16/mambaforge/lib/python3.10/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/Users/mgt16/mambaforge/lib/python3.10/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,


Using TorchDFTD3Calculator for D3 dispersion corrections (see https://github.com/pfnet-research/torch-dftd)
-14.169148097510671


In [ ]:
2+2